In [1]:
puts `ls -l raw-data`


total 9556
-rw-rw-r-- 1 osboxes osboxes   34777 Apr 15 13:32 Demokritos-KG-information.xlsx
-rw-rw-r-- 1 osboxes osboxes  402701 Apr 15 13:32 Disease-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes  125055 Apr 15 13:32 Disease-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes   20901 Apr 15 13:32 Disease-Gene triples.tsv
-rw-rw-r-- 1 osboxes osboxes       0 Apr 15 16:21 disease_list.txt
-rw-rw-r-- 1 osboxes osboxes  207331 Apr 15 13:32 Disease-Therapeutic_Area.tsv
-rw-rw-r-- 1 osboxes osboxes   82213 Apr 15 13:32 Drug-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes 7769112 Apr 15 13:32 Drug-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  111643 Apr 15 13:32 Drug-Drug_type.tsv
-rw-rw-r-- 1 osboxes osboxes   87477 Apr 15 13:32 Drug-Gene triples.tsv
drwxrwxr-x 2 osboxes osboxes    4096 Apr  7 15:55 Feb 2026
-rw-rw-r-- 1 osboxes osboxes  684226 Apr 15 13:32 Gene-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes   57380 Apr 15 13:32 Gene-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  127519 Apr

Disease-Gene triples.tsv COLUMN 6
Drug-Gene triples.tsv COLUMN 6
Gene-Disease triples.tsv COLUMN 2
Gene-Drug triples.tsv  COLUMN 2
Gene-Gene triples.tsv  COLUMN 2
Gene-Pathway triples.tsv COLUMN 2

In [6]:
puts `awk -F'\t' '{print $6}' "raw-data/Disease-Gene triples.tsv" | sort | uniq > gene_list.txt`
puts `cat gene_list.txt | wc -l`


147


In [7]:
puts `awk -F'\t' '{print $6}' "raw-data/Drug-Gene triples.tsv" | sort | uniq >> gene_list.txt`
puts `cat gene_list.txt | wc -l`


592


In [8]:
puts `awk -F'\t' '{print $2}' "raw-data/Gene-Disease triples.tsv" | sort | uniq >> gene_list.txt`
puts `cat gene_list.txt | wc -l`


2732


In [9]:
puts `awk -F'\t' '{print $2}' "raw-data/Gene-Drug triples.tsv" | sort | uniq >> gene_list.txt`
puts `cat gene_list.txt | wc -l`


3095


In [10]:
puts `awk -F'\t' '{print $2}' "raw-data/Gene-Gene triples.tsv" | sort | uniq >> gene_list.txt`
puts `cat gene_list.txt | wc -l`


3725


In [11]:
puts `awk -F'\t' '{print $2}' "raw-data/Gene-Pathway triples.tsv" | sort | uniq >> gene_list.txt`
puts `cat gene_list.txt | wc -l`


3989


In [26]:
puts `(echo "demokritos_gene_cui"; cat gene_list.txt | sort | uniq) > gene_list_uniq.txt`
puts `head -5 gene_list_uniq.txt `
puts `cat gene_list_uniq.txt | wc -l`


demokritos_gene_cui
C0002085
C0017342
C0017361
C0017366
2590


# Column 2 = UMLS CUI

UMLS requires an API key, that has onerous reporting requirements.  Skip it and use mygene.info to find the mapping

need:  source,label,geneid,protein,recommended_full,taxon

## NOTE:  C1418941  is a prion protein that breaks the parser... why?

In [30]:
# puts `curl -H "Accept: application/json" "https://mygene.info/v3/query?q=C1538301&fields=symbol,ensembl.gene,uniprot"`
require "rest-client"
require 'json'

resp =  RestClient.get("https://mygene.info/v3/query?q=C1418941&fields=symbol,name,entrezgene,uniprot,taxid")
puts resp

  data = JSON.parse(resp)
  hit = data.dig('hits', 0)
  genesymbol = hit&.dig('symbol')
  geneid = hit&.dig('entrezgene')
  recommended_full = hit&.dig('name')
  taxon = hit&.dig('taxid')
  protein = hit&.dig('uniprot','Swiss-Prot')

puts protein

{"took":6,"total":1,"max_score":1.7734557,"hits":[{"_id":"5621","_score":1.7734557,"entrezgene":"5621","name":"prion protein (Kanno blood group)","symbol":"PRNP","taxid":9606,"uniprot":{"Swiss-Prot":["P04156","F7VJQ1"],"TrEMBL":"Q53YK7"}}]}
P04156
F7VJQ1


In [ ]:
require "rest-client"
require "json"
require "net/http"
require "uri"

def get_uniprot_protein_name(accession)
  uri = URI("https://rest.uniprot.org/uniprotkb/#{accession}?format=json")
  response = Net::HTTP.get_response(uri)
  return nil unless response.is_a?(Net::HTTPSuccess)
  data = JSON.parse(response.body)
  data.dig('proteinDescription', 'recommendedName', 'fullName', 'value') ||
    data.dig('proteinDescription', 'submissionNames', 0, 'fullName', 'value')
rescue => e
  warn "UniProt lookup failed for #{accession}: #{e}"
  nil
end

def map_cui_to_geneinfo(cui)
  response = RestClient.get("https://mygene.info/v3/query?q=#{cui}&fields=symbol,name,entrezgene,uniprot,taxid")

  data = JSON.parse(response)
  hit = data.dig('hits', 0)
  genesymbol = hit&.dig('symbol')
  geneid     = hit&.dig('entrezgene')
  taxon      = hit&.dig('taxid')
  protein    = hit&.dig('uniprot', 'Swiss-Prot')
  protein    = protein.first if protein.is_a?(Array)  # the prion!  darn it!

  if geneid && protein
    recommended_full = get_uniprot_protein_name(protein) || hit&.dig('name')
    { source: cui, geneid: geneid, genesymbol: genesymbol, recommended_full: recommended_full, taxon: taxon, protein: protein }
  else
    warn "No data found for #{cui}"
    return false
  end
rescue StandardError => e
  warn "Error for #{cui}: #{e.inspect}"
  return false
end

In [31]:
cuis = File.read('gene_list_uniq.txt').split(/\n/)
# cuis[-1]
# cuis[-2]
#cuis[1..-3].first
cuis = cuis[1..-3]  # skip the final two, which are column headers from the original .tsv

["C0002085", "C0017342", "C0017361", "C0017366", "C0017374", "C0017428", "C0017429", "C0022959", "C0024518", "C0029016", "C0029073", "C0033799", "C0035899", "C0079419", "C0079427", "C0079471", "C0079941", "C0085113", "C0085238", "C0086661", "C0162508", "C0242957", "C0242987", "C0242988", "C0249197", "C0282641", "C0314604", "C0376515", "C0376571", "C0376622", "C0440471", "C0525037", "C0537026", "C0598034", "C0599797", "C0600449", "C0678928", "C0678933", "C0678941", "C0694872", "C0694873", "C0694879", "C0694883", "C0694888", "C0694891", "C0694895", "C0694897", "C0694898", "C0751608", "C0751995", "C0812198", "C0812215", "C0812222", "C0812228", "C0812230", "C0812233", "C0812234", "C0812235", "C0812237", "C0812241", "C0812246", "C0812257", "C0812267", "C0812270", "C0812273", "C0812297", "C0812303", "C0812304", "C0812305", "C0812307", "C0812310", "C0812314", "C0872147", "C0879290", "C0879391", "C0879392", "C0879393", "C0879468", "C0879590", "C0919425", "C0919426", "C0919427", "C0919432", "C0

In [33]:
require 'csv'
puts `pwd`


f = File.open('./maps/2026-gene-mappings.map', 'w')
e = File.open('./maps/2026-gene-errors.txt', 'w')
f.sync = true # Ensure immediate writes
e.sync = true # Ensure immediate writes
f.write CSV.generate_line(["source","label","geneid","protein","recommended_full","taxon"])

cuis.each do |cui|

  result = map_cui_to_geneinfo(cui) # {:cui=>"C1538301", :gene_name=>"ATXN3", :ensembl=>"ENSG00000066427"}

  if result == false
    e.write "error getting #{cui}\n"
    next
  end

  warn "CUI: #{cui} #{result[:geneid]} #{result[:genesymbol]}"
#   source: cui, geneid: geneid, genesymbol: genesymbol, recommended_full: recommended_full, taxon: taxon, protein: protein }
  f.write CSV.generate_line([
    cui, 
    result[:genesymbol], 
    "http://purl.uniprot.org/geneid/#{result[:geneid]}",
    "http://purl.uniprot.org/uniprot/#{result[:protein]}",
    result[:recommended_full],
    "http://purl.uniprot.org/taxonomy/#{result[:taxon]}"])
    
end
f.close
e.close
puts "DONE"

/home/osboxes/CODE/SIMPATHIC2/SKG_Mapping/demokritos


No data found for C0002085
No data found for C0017342
No data found for C0017361
No data found for C0017366
No data found for C0017374
No data found for C0017428
No data found for C0017429
No data found for C0022959
No data found for C0024518
No data found for C0029016
No data found for C0029073
No data found for C0033799
No data found for C0035899
CUI: C0079419 7157 TP53
No data found for C0079427
CUI: C0079471 3265 HRAS
No data found for C0079941
CUI: C0085113 4763 NF1
No data found for C0085238
CUI: C0086661 4609 MYC
CUI: C0162508 3725 JUN
CUI: C0242957 2064 ERBB2
CUI: C0242987 7067 THRA
CUI: C0242988 7068 THRB
CUI: C0249197 1026 CDKN1A
No data found for C0282641
No data found for C0314604
CUI: C0376515 596 BCL2
CUI: C0376571 672 BRCA1
CUI: C0376622 5243 ABCB1
No data found for C0440471
CUI: C0525037 1029 CDKN2A
CUI: C0537026 54658 UGT1A1
CUI: C0598034 675 BRCA2
No data found for C0599797
No data found for C0600449
No data found for C0678928
No data found for C0678933
No data found 

DONE


In [1]:
# Patch cell: replace recommended_full with UniProt protein recommended names.
# The core mapping cell (above) populates recommended_full from mygene.info's 'name'
# field, which is the NCBI gene description. This patch calls UniProt directly for
# each protein accession already in the map and overwrites that column with the
# canonical UniProt protein recommended name instead.

require 'net/http'
require 'uri'
require 'json'
require 'csv'

MAP_FILE   = './maps/2026-gene-mappings.map'
BATCH_SIZE = 50

rows = CSV.read(MAP_FILE, headers: true).map(&:to_h)
puts "Loaded #{rows.size} rows"

# Build unique accession list from the protein column
accessions = rows.map { |r| r['protein'].to_s.match(/uniprot\/([A-Z0-9]+)/)&.[](1) }.compact.uniq
puts "#{accessions.size} unique UniProt accessions to fetch"

acc_to_name = {}

accessions.each_slice(BATCH_SIZE).with_index(1) do |batch, i|
  query = batch.map { |a| "accession:#{a}" }.join(' OR ')
  uri   = URI('https://rest.uniprot.org/uniprotkb/search')
  uri.query = URI.encode_www_form(
    query:  query,
    fields: 'accession,protein_name',
    format: 'json',
    size:   BATCH_SIZE + 10
  )

  resp = Net::HTTP.get_response(uri)
  unless resp.is_a?(Net::HTTPSuccess)
    warn "Batch #{i} failed: HTTP #{resp.code}"
    next
  end

  JSON.parse(resp.body)['results'].each do |entry|
    acc  = entry['primaryAccession']
    name = entry.dig('proteinDescription', 'recommendedName', 'fullName', 'value') ||
           entry.dig('proteinDescription', 'submissionNames', 0, 'fullName', 'value')
    acc_to_name[acc] = name if name
  end

  warn "Batch #{i}/#{(accessions.size.to_f / BATCH_SIZE).ceil} done — #{acc_to_name.size} names collected"
  sleep 0.2
end

puts "Fetched #{acc_to_name.size} protein names from UniProt"

# Update recommended_full in each row
no_match = []
rows.each do |row|
  acc = row['protein'].to_s.match(/uniprot\/([A-Z0-9]+)/)&.[](1)
  next unless acc
  if (name = acc_to_name[acc])
    row['recommended_full'] = name
  else
    no_match << acc
  end
end

warn "No UniProt name found for: #{no_match.join(', ')}" unless no_match.empty?

CSV.open(MAP_FILE, 'w', write_headers: true, headers: rows.first.keys) do |csv|
  rows.each { |row| csv << row.values }
end

puts "Done — #{rows.size - no_match.size}/#{rows.size} rows updated in #{MAP_FILE}"

Loaded 2377 rows
2377 unique UniProt accessions to fetch


Batch 1/48 done — 50 names collected
Batch 2/48 done — 100 names collected
Batch 3/48 done — 150 names collected
Batch 4/48 done — 200 names collected
Batch 5/48 done — 250 names collected
Batch 6/48 done — 300 names collected
Batch 7/48 done — 350 names collected
Batch 8/48 done — 400 names collected
Batch 9/48 done — 450 names collected
Batch 10/48 done — 500 names collected
Batch 11/48 done — 550 names collected
Batch 12/48 done — 600 names collected
Batch 13/48 done — 650 names collected
Batch 14/48 done — 700 names collected
Batch 15/48 done — 750 names collected
Batch 16/48 done — 800 names collected
Batch 17/48 done — 850 names collected
Batch 18/48 done — 900 names collected
Batch 19/48 done — 950 names collected
Batch 20/48 done — 1000 names collected
Batch 21/48 done — 1050 names collected
Batch 22/48 done — 1100 names collected
Batch 23/48 done — 1150 names collected
Batch 24/48 done — 1200 names collected
Batch 25/48 done — 1250 names collected
Batch 26/48 done — 1300 names

Fetched 2377 protein names from UniProt
Done — 2377/2377 rows updated in ./maps/2026-gene-mappings.map
